In [0]:
--============================================================
--SILVER: Cleaned, typed, deduplicated, validated
--One MERGE INTO per entity — idempotent, safe to re-run.
--===========================================================
USE CATALOG f1_project;

------------ dim-ready: circuits ----------
CREATE TABLE IF NOT EXISTS f1_project.silver.circuits_clean (
  circuit_id INT, circuit_ref STRING, name STRING, location STRING,
  country STRING, lat DOUBLE, lng DOUBLE, alt INT
);


MERGE INTO f1_project.silver.circuits_clean AS target
USING (
  SELECT DISTINCT
    CAST(circuit_id AS INT) AS circuit_id,
    circuit_ref, TRIM(name) AS name, TRIM(location) AS location,
    TRIM(country) AS country,
    TRY_CAST(lat AS DOUBLE) AS lat, TRY_CAST(lng AS DOUBLE) AS lng,
    TRY_CAST(alt AS INT) AS alt
  FROM f1_project.bronze.circuits_raw
  WHERE circuit_id IS NOT NULL
) AS source
ON target.circuit_id = source.circuit_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;


-- ---------- dim-ready: races ----------
CREATE TABLE IF NOT EXISTS f1_project.silver.races_clean (
  race_id INT, year INT, round INT, circuit_id INT,
  name STRING, race_date DATE, race_time STRING
);


MERGE INTO f1_project.silver.races_clean AS target
USING (
  SELECT DISTINCT
    CAST(race_id AS INT) AS race_id, CAST(year AS INT) AS year,
    CAST(round AS INT) AS round, CAST(circuit_id AS INT) AS circuit_id,
    TRIM(name) AS name, TRY_CAST(date AS DATE) AS race_date, time AS race_time
  FROM f1_project.bronze.races_raw
  WHERE race_id IS NOT NULL AND TRY_CAST(date AS DATE) IS NOT NULL
) AS source
ON target.race_id = source.race_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;



-- ---------- dim-ready: drivers ----------
CREATE TABLE IF NOT EXISTS f1_project.silver.drivers_clean (
  driver_id INT, driver_ref STRING, number INT, code STRING,
  forename STRING, surname STRING, full_name STRING, dob DATE, nationality STRING
);


MERGE INTO f1_project.silver.drivers_clean AS target
USING (
  SELECT DISTINCT
    CAST(driver_id AS INT) AS driver_id, driver_ref,
    TRY_CAST(number AS INT) AS number, code,
    INITCAP(TRIM(forename)) AS forename, INITCAP(TRIM(surname)) AS surname,
    CONCAT(INITCAP(TRIM(forename)), ' ', INITCAP(TRIM(surname))) AS full_name,
    TRY_CAST(dob AS DATE) AS dob, TRIM(nationality) AS nationality
  FROM f1_project.bronze.drivers_raw
  WHERE driver_id IS NOT NULL
) AS source
ON target.driver_id = source.driver_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

-- ---------- dim-ready: constructors ----------

CREATE TABLE IF NOT EXISTS f1_project.silver.constructors_clean (
  constructor_id INT, constructor_ref STRING, name STRING, nationality STRING
);


MERGE INTO f1_project.silver.constructors_clean AS target
USING (
  SELECT DISTINCT
    CAST(constructor_id AS INT) AS constructor_id, constructor_ref,
    TRIM(name) AS name, TRIM(nationality) AS nationality
  FROM f1_project.bronze.constructors_raw
  WHERE constructor_id IS NOT NULL
) AS source
ON target.constructor_id = source.constructor_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

-- ---------- fact-ready: results ----------
-- Data quality rules applied INSIDE the source query, not after —
-- this avoids the "constraint violated by inserted row" trap.
CREATE TABLE IF NOT EXISTS f1_project.silver.results_clean (
  result_id INT, race_id INT, driver_id INT, constructor_id INT,
  grid INT, position INT, points INT, laps INT, result_time STRING,
  fastest_lap INT, rank INT, fastest_lap_time STRING, status STRING
);


MERGE INTO f1_project.silver.results_clean AS target
USING (
  SELECT DISTINCT
    CAST(result_id AS INT) AS result_id, CAST(race_id AS INT) AS race_id,
    CAST(driver_id AS INT) AS driver_id, CAST(constructor_id AS INT) AS constructor_id,
    TRY_CAST(grid AS INT) AS grid, TRY_CAST(position AS INT) AS position,
    TRY_CAST(points AS INT) AS points, TRY_CAST(laps AS INT) AS laps,
    time AS result_time, TRY_CAST(fastest_lap AS INT) AS fastest_lap,
    TRY_CAST(rank AS INT) AS rank, fastest_lap_time, TRIM(status) AS status
  FROM f1_project.bronze.results_raw
  WHERE result_id IS NOT NULL
    AND race_id IS NOT NULL
    AND driver_id IS NOT NULL
    AND (TRY_CAST(points AS INT) IS NULL OR TRY_CAST(points AS INT) >= 0)  -- drop negative-point bad rows
) AS source
ON target.result_id = source.result_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

-- ---------- fact-ready: pit_stops ----------
CREATE TABLE IF NOT EXISTS f1_project.silver.pit_stops_clean (
  race_id INT, driver_id INT, stop INT, lap INT, stop_time STRING, duration DECIMAL(6,3)
);


MERGE INTO f1_project.silver.pit_stops_clean AS target
USING (
  SELECT DISTINCT
    CAST(race_id AS INT) AS race_id, CAST(driver_id AS INT) AS driver_id,
    CAST(stop AS INT) AS stop, CAST(lap AS INT) AS lap,
    time AS stop_time, TRY_CAST(duration AS DECIMAL(6,3)) AS duration
  FROM f1_project.bronze.pit_stops_raw
  WHERE race_id IS NOT NULL AND driver_id IS NOT NULL
    AND TRY_CAST(duration AS DECIMAL(6,3)) > 0
) AS source
ON target.race_id = source.race_id AND target.driver_id = source.driver_id AND target.stop = source.stop
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

-- ---------- fact-ready: lap_times ----------
CREATE TABLE IF NOT EXISTS f1_project.silver.lap_times_clean (
  race_id INT, driver_id INT, lap INT, position INT, lap_time STRING, milliseconds INT
);


MERGE INTO f1_project.silver.lap_times_clean AS target
USING (
  SELECT DISTINCT
    CAST(race_id AS INT) AS race_id, CAST(driver_id AS INT) AS driver_id,
    CAST(lap AS INT) AS lap, TRY_CAST(position AS INT) AS position,
    time AS lap_time, TRY_CAST(milliseconds AS INT) AS milliseconds
  FROM f1_project.bronze.lap_times_raw
  WHERE race_id IS NOT NULL AND driver_id IS NOT NULL AND lap IS NOT NULL
) AS source
ON target.race_id = source.race_id AND target.driver_id = source.driver_id AND target.lap = source.lap
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

-- ---------- fact-ready: qualifying ----------
CREATE TABLE IF NOT EXISTS f1_project.silver.qualifying_clean (
  qualify_id INT, race_id INT, driver_id INT, constructor_id INT,
  number INT, position INT, q1 STRING, q2 STRING, q3 STRING
);


MERGE INTO f1_project.silver.qualifying_clean AS target
USING (
  SELECT DISTINCT
    CAST(qualify_id AS INT) AS qualify_id, CAST(race_id AS INT) AS race_id,
    CAST(driver_id AS INT) AS driver_id, CAST(constructor_id AS INT) AS constructor_id,
    TRY_CAST(number AS INT) AS number, TRY_CAST(position AS INT) AS position,
    q1, q2, q3
  FROM f1_project.bronze.qualifying_raw
  WHERE qualify_id IS NOT NULL
) AS source
ON target.qualify_id = source.qualify_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

-- ---------- Data quality constraints ----------
ALTER TABLE f1_project.silver.circuits_clean ALTER COLUMN circuit_id SET NOT NULL;
ALTER TABLE f1_project.silver.circuits_clean ADD CONSTRAINT pk_circuit_id PRIMARY KEY (circuit_id);

ALTER TABLE f1_project.silver.races_clean ALTER COLUMN race_id SET NOT NULL;
ALTER TABLE f1_project.silver.races_clean ADD CONSTRAINT pk_race_id PRIMARY KEY (race_id);

ALTER TABLE f1_project.silver.drivers_clean ALTER COLUMN driver_id SET NOT NULL;
ALTER TABLE f1_project.silver.drivers_clean ADD CONSTRAINT pk_driver_id PRIMARY KEY (driver_id);

ALTER TABLE f1_project.silver.constructors_clean ALTER COLUMN constructor_id SET NOT NULL;
ALTER TABLE f1_project.silver.constructors_clean ADD CONSTRAINT pk_constructor_id PRIMARY KEY (constructor_id);

ALTER TABLE f1_project.silver.results_clean ALTER COLUMN result_id SET NOT NULL;
ALTER TABLE f1_project.silver.results_clean ADD CONSTRAINT pk_result_id PRIMARY KEY (result_id);
ALTER TABLE f1_project.silver.results_clean ADD CONSTRAINT valid_points CHECK (points IS NULL OR points >= 0);

-- ---------- Sanity checks ----------
SELECT 'circuits' AS tbl, count(*) AS cnt FROM f1_project.silver.circuits_clean
UNION ALL SELECT 'races', count(*) FROM f1_project.silver.races_clean
UNION ALL SELECT 'drivers', count(*) FROM f1_project.silver.drivers_clean
UNION ALL SELECT 'constructors', count(*) FROM f1_project.silver.constructors_clean
UNION ALL SELECT 'results', count(*) FROM f1_project.silver.results_clean
UNION ALL SELECT 'pit_stops', count(*) FROM f1_project.silver.pit_stops_clean
UNION ALL SELECT 'lap_times', count(*) FROM f1_project.silver.lap_times_clean
UNION ALL SELECT 'qualifying', count(*) FROM f1_project.silver.qualifying_clean;
